## `EDA (Exploratory Data Analysis)`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [2]:
sns.set_theme(style="whitegrid")

In [3]:
df = pd.read_csv('../data/Anuvaad_INDB.csv')

In [4]:
display(df.head())

,food_code,food_name,primarysource,energy_kj,energy_kcal,carb_g,protein_g,fat_g,freesugar_g,fibre_g,...,unit_serving_folate_ug,unit_serving_vitb1_mg,unit_serving_vitb2_mg,unit_serving_vitb3_mg,unit_serving_vitb5_mg,unit_serving_vitb6_mg,unit_serving_vitb7_ug,unit_serving_vitb9_ug,unit_serving_vitc_mg,unit_serving_carotenoids_ug
0,ASC001,Hot tea (Garam Chai),asc_manual,68.16,16.14,2.58,0.39,0.53,2.58,0.00,...,1.80,0.01,0.03,0.02,0.09,0.01,0.51,1.80,0.50,50.00
1,ASC002,Instant coffee,asc_manual,97.77,23.16,3.65,0.64,0.75,3.62,0.00,...,5.60,0.02,0.09,0.80,0.25,0.03,3.49,5.60,1.51,150.00
2,ASC003,Espreso coffee,asc_manual,217.00,51.54,6.62,1.75,2.14,6.53,0.00,...,5.53,0.02,0.09,0.56,0.26,0.03,2.87,5.53,1.51,150.00
3,ASC004,Iced tea,asc_manual,44.09,10.34,2.70,0.03,0.01,2.70,0.00,...,1.28,0.01,0.00,0.02,0.02,0.01,0.02,1.28,5.95,0.70
4,ASC005,Raw mango drink (Aam panna),asc_manual,152.39,35.92,9.05,0.16,0.03,7.49,0.61,...,14.05,0.01,0.01,0.14,0.07,0.07,0.73,14.05,45.30,448.89


In [5]:
print("\n--- Missing Values Check ---")
print(df.isnull().sum())


--- Missing Values Check ---
food_code                       0
food_name                       0
primarysource                   0
energy_kj                       0
energy_kcal                     0
                               ..
unit_serving_vitb6_mg          77
unit_serving_vitb7_ug          77
unit_serving_vitb9_ug          77
unit_serving_vitc_mg           77
unit_serving_carotenoids_ug    77
Length: 82, dtype: int64


In [6]:
critical_cols = [
    'unit_serving_fibre_g', 'unit_serving_potassium_mg', 'unit_serving_magnesium_mg', 
    'unit_serving_calcium_mg', 'unit_serving_protein_g', 'unit_serving_sodium_mg', 
    'unit_serving_sfa_mg', 'unit_serving_freesugar_g'
]

print(f"Total dishes before cleaning: {len(df)}")

df.dropna(subset=critical_cols, inplace=True)

print(f"Total dishes after dropping invalid rows: {len(df)}")


Total dishes before cleaning: 1004
Total dishes after dropping invalid rows: 927


In [7]:
# --- PHASE 1: The Hard Filter ---
# Eliminate any single serving with more than 800mg of sodium
unsafe_threshold = 800
safe_df = df[df['unit_serving_sodium_mg'] <= unsafe_threshold].copy()

print(f"Safe, clinically valid meals for the engine: {len(safe_df)}")

Safe, clinically valid meals for the engine: 852


## `PHASE 2: THE DASH SCORER`


In [8]:
metrics_to_scale = [
    'unit_serving_fibre_g', 'unit_serving_potassium_mg', 'unit_serving_magnesium_mg', 
    'unit_serving_calcium_mg', 'unit_serving_protein_g', 'unit_serving_sodium_mg', 
    'unit_serving_sfa_mg', 'unit_serving_freesugar_g'
]

# Apply Z-Score Standardization
scaler = StandardScaler()
safe_df_scaled = safe_df.copy()
safe_df_scaled[metrics_to_scale] = scaler.fit_transform(safe_df[metrics_to_scale])

def calculate_zscore_dash(row):
    """
    Weights approximate the combined protective effect of High K + High Mg + Adequate Ca against Na.
    """
    # REWARDS (Cardiovascular Promoters)
    fiber_pts = row['unit_serving_fibre_g'] * 5.0          
    potassium_pts = row['unit_serving_potassium_mg'] * 3.5 
    
    magnesium_pts = row['unit_serving_magnesium_mg'] * 2.0 
    calcium_pts = row['unit_serving_calcium_mg'] * 2.0     
    protein_pts = row['unit_serving_protein_g'] * 1.0      
    
    # PENALTIES (Hypertension Exacerbators)
    sodium_pen = row['unit_serving_sodium_mg'] * 4.0       
    sat_fat_pen = row['unit_serving_sfa_mg'] * 3.0         
    sugar_pen = row['unit_serving_freesugar_g'] * 2.0      
    
    rewards = fiber_pts + potassium_pts + magnesium_pts + calcium_pts + protein_pts
    penalties = sodium_pen + sat_fat_pen + sugar_pen
    
    return round((rewards - penalties), 3)

# Apply the score and rank
safe_df['DASH_Score'] = safe_df_scaled.apply(calculate_zscore_dash, axis=1)
ranked_meals = safe_df.sort_values(by='DASH_Score', ascending=False)

# Display the top 5 results
display_cols = [
    'food_name', 'DASH_Score', 'unit_serving_potassium_mg', 
    'unit_serving_sodium_mg', 'unit_serving_fibre_g', 'unit_serving_sfa_mg'
]

print("\n--- TOP 5 CLINICALLY OPTIMAL INDIAN MEALS (K-Adjusted) ---")
display(ranked_meals[display_cols].head(5))


--- TOP 5 CLINICALLY OPTIMAL INDIAN MEALS (K-Adjusted) ---


,food_name,DASH_Score,unit_serving_potassium_mg,unit_serving_sodium_mg,unit_serving_fibre_g,unit_serving_sfa_mg
210,Spinach mutton (Palak mutton),53.128,2256.47,588.57,10.85,1927.08
512,Lasagne with vegetables,45.558,2221.88,716.66,11.66,18624.06
800,Lotus seed halwa (Kamal gattay ka halwa),42.535,2387.13,11.08,1.53,40028.29
798,Kiwi granola pudding,42.116,1438.54,310.28,19.44,29701.95
201,Avial,37.204,1267.49,273.05,18.24,20865.47


## `PHASE 3: METABOLIC ENGINE & PLAN GENERATOR`


In [9]:
def calculate_dash_tdee_and_constraints(weight_kg, height_cm, age_years, gender, activity_level, bp_stage):
    """
    Calculates TDEE with Indian adjustment + DASH constraints for hypertensive patients.
    *NOTE: Intentionally leaves a 10% unallocated sodium buffer for real-world cooking variables.*
    """
    # Mifflin-St Jeor BMR
    if gender.lower() == 'male':
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years + 5
    else:  
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years - 161
        
    # Activity multiplier
    activity_multipliers = {'sedentary': 1.2, 'light': 1.375, 'moderate': 1.55, 'active': 1.725}
    tdee = bmr * activity_multipliers.get(activity_level, 1.2)
    
    #Indian BMR adjustment (ICMR/NIN standards)
    indian_tdee = tdee * 0.95
    
    # Step 4: BP-based sodium tiers (AHA/DASH)
    sodium_daily = {'pre': 2300, 'stage1': 2000, 'stage2': 1500}
    dash_sodium_limit = sodium_daily[bp_stage]
    
    # Meal breakdowns (3 main + 2 snacks)
    meals = {
        'Breakfast': round(indian_tdee * 0.25, 0),
        'Lunch': round(indian_tdee * 0.30, 0),
        'Dinner': round(indian_tdee * 0.30, 0),
        'Snack 1': round(indian_tdee * 0.075, 0),
        'Snack 2': round(indian_tdee * 0.075, 0)
    }
    
    # Per-meal sodium max (Allocating 90% total, leaving a 10% clinical safety buffer)
    sodium_per_meal = {
        'main': dash_sodium_limit * 0.25,
        'snack': dash_sodium_limit * 0.075
    }
    
    return {
        'indian_tdee': round(indian_tdee, 0),
        'meal_calories': meals,
        'sodium_per_main': round(sodium_per_meal['main'], 0),
        'sodium_per_snack': round(sodium_per_meal['snack'], 0),
        'daily_sodium_limit': dash_sodium_limit
    }


def generate_full_day_plan(ranked_meals_df, constraints):
    """
    Greedy algorithm that fills 5 specific daily meal slots using the highest-scoring 
    DASH meals that fit the specific calorie and sodium limits for each slot.
    """
    daily_plan = []
    used_indices = set() # Prevents prescribing the same meal twice in one day
    
    total_cals = 0
    total_sodium = 0

    slots = [
        ('Breakfast', constraints['meal_calories']['Breakfast'], constraints['sodium_per_main'], 100),
        ('Lunch', constraints['meal_calories']['Lunch'], constraints['sodium_per_main'], 150),
        ('Dinner', constraints['meal_calories']['Dinner'], constraints['sodium_per_main'], 150),
        ('Snack 1', constraints['meal_calories']['Snack 1'], constraints['sodium_per_snack'], 50),
        ('Snack 2', constraints['meal_calories']['Snack 2'], constraints['sodium_per_snack'], 50)
    ]
    
    for slot_name, target_cal, max_na, tolerance in slots:
        best_meal = None
        
        # Search the ranked list from top (healthiest) to bottom
        for idx, meal in ranked_meals_df.iterrows():
            if idx in used_indices:
                continue
                
            meal_cal = meal['unit_serving_energy_kcal']
            meal_na = meal['unit_serving_sodium_mg']
            
            # If the meal fits the calorie window AND is under the strict sodium limit
            if (target_cal - tolerance <= meal_cal <= target_cal + tolerance) and (meal_na <= max_na):
                best_meal = meal.copy()
                best_meal['Meal_Slot'] = slot_name
                best_meal['Target_Cal'] = target_cal
                
                used_indices.add(idx)
                daily_plan.append(best_meal)
                
                total_cals += meal_cal
                total_sodium += meal_na
                break
                
        if best_meal is None:
            print(f"Warning: Could not find a perfect match for {slot_name}. Consider widening the calorie tolerance.")

    return pd.DataFrame(daily_plan), total_cals, total_sodium

In [10]:
patient_profile = {
    'weight_kg': 75, 'height_cm': 165, 'age_years': 55, 
    'gender': 'male', 'activity_level': 'sedentary', 'bp_stage': 'stage2'
}

# 2. Calculate his specific metabolic and clinical constraints
constraints = calculate_dash_tdee_and_constraints(**patient_profile)

print("--- TEST PATIENT: RAJESH (STAGE 2 HTN) ---")
print(f"Goal Calories: {constraints['indian_tdee']} kcal/day")
print(f"Goal Sodium: < {constraints['daily_sodium_limit']} mg/day (10% Buffer Active)\n")

# 3. Generate the Plan using Phase 2's 'ranked_meals' dataframe
plan_df, achieved_cals, achieved_na = generate_full_day_plan(ranked_meals, constraints)

# 4. Display the Output beautifully
display_cols = [
    'Meal_Slot', 'food_name', 'DASH_Score', 
    'unit_serving_energy_kcal', 'unit_serving_sodium_mg', 'unit_serving_potassium_mg'
]

print("--- PRESCRIBED DAILY DIET PLAN ---")
# Reorder columns for readability
display(plan_df[display_cols].set_index('Meal_Slot'))

print("\n--- DAILY CLINICAL TOTALS ---")
print(f"Total Calories Prescribed: {round(achieved_cals)} kcal")
print(f"Total Sodium Prescribed: {round(achieved_na)} mg")
print(f"Total Sodium Buffer Remaining: {round(constraints['daily_sodium_limit'] - achieved_na)} mg (Safe)")

--- TEST PATIENT: RAJESH (STAGE 2 HTN) ---
Goal Calories: 1723.0 kcal/day
Goal Sodium: < 1500 mg/day (10% Buffer Active)

--- PRESCRIBED DAILY DIET PLAN ---


,food_name,DASH_Score,unit_serving_energy_kcal,unit_serving_sodium_mg,unit_serving_potassium_mg
Meal_Slot,,,,,
Breakfast,Avial,37.204,417.34,273.05,1267.49
Lunch,Apple oats chia seed smoothie,34.400,538.28,201.25,1063.76
Dinner,Eggplant/Brinjal rice (Vangi bhat),32.638,538.09,171.96,889.07
Snack 1,Sajina,7.319,172.63,5.85,291.25
Snack 2,Methi parantha/paratha,7.162,135.30,44.10,176.85



--- DAILY CLINICAL TOTALS ---
Total Calories Prescribed: 1802 kcal
Total Sodium Prescribed: 696 mg
Total Sodium Buffer Remaining: 804 mg (Safe)


In [12]:
def calculate_dash_tdee_and_constraints(weight_kg, height_cm, age_years, gender, activity_level, bp_stage):
    """
    Standard Mifflin-St Jeor with Indian Metabolic Adjustment (-5%) 
    and Stage-specific Sodium Tiers.
    """
    if gender.lower() == 'male':
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years + 5
    else:  
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years - 161
        
    multipliers = {'sedentary': 1.2, 'light': 1.375, 'moderate': 1.55, 'active': 1.725}
    indian_tdee = (bmr * multipliers.get(activity_level, 1.2)) * 0.95
    
    sodium_daily = {'pre': 2300, 'stage1': 2000, 'stage2': 1500}
    dash_limit = sodium_daily[bp_stage]
    
    # Target Calories per slot
    meals = {
        'Breakfast': round(indian_tdee * 0.25),
        'Lunch': round(indian_tdee * 0.30),
        'Dinner': round(indian_tdee * 0.30),
        'Snack': round(indian_tdee * 0.075) 
    }
    
    # 10% Clinical Sodium Buffer Strategy
    return {
        'indian_tdee': round(indian_tdee),
        'meal_calories': meals,
        'sodium_per_main': round(dash_limit * 0.25),
        'sodium_per_snack': round(dash_limit * 0.075),
        'daily_sodium_limit': dash_limit
    }

def generate_exchange_plan(ranked_df, constraints, options_per_slot=3):
    """
    Generates a 'Gallery' of choices for each meal slot.
    Features a Global Memory Tracker to ensure no dish is repeated across different slots.
    """
    slots = [
        ('Breakfast', constraints['meal_calories']['Breakfast'], constraints['sodium_per_main'], 100),
        ('Lunch', constraints['meal_calories']['Lunch'], constraints['sodium_per_main'], 150),
        ('Dinner', constraints['meal_calories']['Dinner'], constraints['sodium_per_main'], 150),
        ('Snack 1', constraints['meal_calories']['Snack'], constraints['sodium_per_snack'], 50),
        ('Snack 2', constraints['meal_calories']['Snack'], constraints['sodium_per_snack'], 50)
    ]
    
    full_exchange_list = []
    used_indices = set() # GLOBAL memory tracker 
    
    for slot_name, target_cal, max_na, tolerance in slots:
        slot_options = []
        
        # Iterate through our ranked meals
        for idx, meal in ranked_df.iterrows():
            # Stop if we found our 3 options for this specific slot
            if len(slot_options) >= options_per_slot:
                break
                
            # Skip if this meal was already used in an earlier meal slot
            if idx in used_indices:
                continue
                
            meal_cal = meal['unit_serving_energy_kcal']
            meal_na = meal['unit_serving_sodium_mg']
            
            # If it fits the calorie window AND is safe for sodium
            if (target_cal - tolerance <= meal_cal <= target_cal + tolerance) and (meal_na <= max_na):
                best_meal = meal.copy()
                best_meal['Meal_Slot'] = slot_name
                best_meal['Target_Kcal'] = target_cal
                
                slot_options.append(best_meal)
                used_indices.add(idx) # Mark this meal as "used" globally!
                
        if slot_options:
            full_exchange_list.append(pd.DataFrame(slot_options))
            
    return pd.concat(full_exchange_list)


In [13]:
# Example: Rajesh (Stage 2 Hypertension)
rajesh_profile = {
    'weight_kg': 75, 'height_cm': 165, 'age_years': 55, 
    'gender': 'male', 'activity_level': 'sedentary', 'bp_stage': 'stage2'
}

# 1. Get Constraints
limits = calculate_dash_tdee_and_constraints(**rajesh_profile)

# 2. Generate Top 3 Options for each slot
exchange_plan = generate_exchange_plan(ranked_meals, limits)

# 3. Clean Display
view_cols = ['Meal_Slot', 'food_name', 'DASH_Score', 'unit_serving_energy_kcal', 'unit_serving_sodium_mg']
print(f"--- CLINICAL DIET GALLERY FOR RAJESH ({limits['indian_tdee']} kcal) ---")
display(exchange_plan[view_cols].groupby('Meal_Slot').head(3))

--- CLINICAL DIET GALLERY FOR RAJESH (1723 kcal) ---


,Meal_Slot,food_name,DASH_Score,unit_serving_energy_kcal,unit_serving_sodium_mg
201,Breakfast,Avial,37.204,417.34,273.05
810,Breakfast,Kidney bean sandwich with cottage cheese,25.933,423.42,234.84
503,Breakfast,Masala dosa mixed vegetable fillings,25.310,400.84,338.57
787,Lunch,Apple oats chia seed smoothie,34.400,538.28,201.25
491,Lunch,Eggplant/Brinjal rice (Vangi bhat),32.638,538.09,171.96
826,Lunch,Pear preserves (Naashpati ka murabba),16.950,603.53,12.52
876,Dinner,Vegetable namkeen jave,16.779,409.97,117.36
223,Dinner,Butter chicken,15.296,377.04,72.14
194,Dinner,Shepherd's pie (vegetarian),14.435,367.11,319.92
746,Snack 1,Sajina,7.319,172.63,5.85
